# Seattle Building Permits — Exploratory Data Analysis

This notebook pulls the Seattle Building Permits dataset from the Seattle Open Data portal and explores it with a focus on accessory dwelling units (ADUs). The goal is to understand what people are actually building in Seattle backyards — which is often clearer than reading the municipal code.

**Dataset:** [Seattle Building Permits](https://data.seattle.gov/Permitting/Building-Permits/76t5-zqzr)  
**Source:** Seattle Open Data portal (Socrata API)  
**Output:** `data/raw/seattle_building_permits.csv`

## 1. Setup

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RAW_DATA_PATH = Path("../data/raw/seattle_building_permits.csv")
SOCRATA_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
DATASET_ID = "76t5-zqzr"
BASE_URL = f"https://data.seattle.gov/resource/{DATASET_ID}.json"

## 2. Download dataset

Pull the full dataset from the Socrata API using pagination (500 rows per request). Skips the download if the file already exists locally.

In [ ]:
def fetch_all_records(base_url: str, token: str | None, limit: int = 50000) -> list[dict]:
    """Page through the Socrata API and return all records."""
    headers = {"X-App-Token": token} if token else {}
    records = []
    offset = 0

    while True:
        resp = requests.get(
            base_url,
            headers=headers,
            params={"$limit": limit, "$offset": offset},
        )
        resp.raise_for_status()
        batch = resp.json()
        records.extend(batch)
        print(f"  fetched {len(records):,} records...", end="\r")
        if len(batch) < limit:
            break
        offset += limit

    print(f"\nDone — {len(records):,} total records")
    return records


if RAW_DATA_PATH.exists():
    print(f"File already exists at {RAW_DATA_PATH} — skipping download")
else:
    print("Fetching Seattle Building Permits...")
    records = fetch_all_records(BASE_URL, SOCRATA_TOKEN)
    df_raw = pd.DataFrame(records)
    RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(RAW_DATA_PATH, index=False)
    print(f"Saved to {RAW_DATA_PATH}")

In [ ]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

## 3. Schema inspection

Profile every column: type, null rate, unique value count, and sample values. Anything surprising gets flagged — these observations feed directly into the dbt staging model and test design.

In [ ]:
# Column-level profile: type, null rate, unique count, sample values
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "unique_count": df.nunique(),
    "sample_values": [df[c].dropna().unique()[:3].tolist() for c in df.columns],
})
profile

In [ ]:
# Columns with high null rates (>50%) — flag as unreliable for dbt tests
high_null = profile[profile["null_pct"] > 50].sort_values("null_pct", ascending=False)
print(f"{len(high_null)} columns with >50% nulls:")
high_null[["dtype", "null_pct"]]

In [ ]:
# Date columns — check parsing and range
date_cols = [c for c in df.columns if "date" in c.lower()]
for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    valid = parsed.dropna()
    print(f"{col}: {valid.min().date()} → {valid.max().date()} ({parsed.isnull().sum():,} unparseable)")

In [ ]:
# Permit type distribution — what categories exist?
permit_type_col = next((c for c in df.columns if "permittype" in c.lower() or "permit_type" in c.lower()), None)
print(f"Using column: {permit_type_col}")
df[permit_type_col].value_counts().head(20)

## 4. Identify ADU-relevant permit types and codes

Search for DADU, ADU, backyard cottage, and related terms across permit type, description, and category fields. The goal is to isolate the exact filters needed to define an "ADU permit" for all downstream analysis.

In [ ]:
# Terms to search across all string columns
ADU_TERMS = ["adu", "dadu", "accessory dwelling", "backyard cottage", "in-law", "carriage house"]

# Identify string columns to search
str_cols = df.select_dtypes(include="object").columns.tolist()

# Search each column for any ADU term (case-insensitive)
hits = {}
for col in str_cols:
    mask = df[col].str.lower().str.contains("|".join(ADU_TERMS), na=False)
    if mask.any():
        hits[col] = df.loc[mask, col].value_counts().head(10)
        print(f"✓ '{col}' — {mask.sum():,} matches")

print(f"\nTotal columns with ADU-related content: {len(hits)}")

In [ ]:
# Show top matching values per column
for col, counts in hits.items():
    print(f"\n--- {col} ---")
    print(counts.to_string())

In [ ]:
# Build the ADU filter — update this based on what the search reveals above
# This is the canonical definition used in all downstream analysis
adu_mask = df[str_cols].apply(
    lambda col: col.str.lower().str.contains("|".join(ADU_TERMS), na=False)
).any(axis=1)

df_adu = df[adu_mask].copy()
print(f"ADU permits identified: {len(df_adu):,} ({len(df_adu)/len(df)*100:.1f}% of all permits)")

## 5. Volume and trend analysis

Basic charts on the ADU permit subset: applications by year, by neighbourhood, approval timelines, and value of work distributions. These become the core visuals for Substack Post 1.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

# Parse application date
app_date_col = next((c for c in df_adu.columns if "application" in c.lower() and "date" in c.lower()), None)
df_adu["application_date"] = pd.to_datetime(df_adu[app_date_col], errors="coerce")
df_adu["year"] = df_adu["application_date"].dt.year

print(f"Application date column: {app_date_col}")

In [ ]:
# ADU permits by year
yearly = df_adu.groupby("year").size().reset_index(name="count")
yearly = yearly[yearly["year"] >= 2000]  # filter out data quality outliers

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly["year"], yearly["count"], color=sns.color_palette("muted")[0])
ax.set_title("Seattle ADU Permit Applications by Year", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Number of permits")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

In [ ]:
# Top neighbourhoods by ADU permit volume
neighbourhood_col = next((c for c in df_adu.columns if "neighbour" in c.lower() or "neighborhood" in c.lower() or "neighbourhood" in c.lower()), None)

if neighbourhood_col:
    top_neighbourhoods = df_adu[neighbourhood_col].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    top_neighbourhoods.plot(kind="barh", ax=ax, color=sns.color_palette("muted")[1])
    ax.set_title("Top 15 Neighbourhoods by ADU Permit Volume", fontsize=14)
    ax.set_xlabel("Number of permits")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No neighbourhood column found — check column names:", [c for c in df_adu.columns if "zip" in c.lower() or "geo" in c.lower()])

In [ ]:
# Approval timeline: days from application to issue
issue_date_col = next((c for c in df_adu.columns if "issue" in c.lower() and "date" in c.lower()), None)

if issue_date_col:
    df_adu["issue_date"] = pd.to_datetime(df_adu[issue_date_col], errors="coerce")
    df_adu["days_to_approval"] = (df_adu["issue_date"] - df_adu["application_date"]).dt.days
    valid_timelines = df_adu["days_to_approval"].dropna()
    valid_timelines = valid_timelines[(valid_timelines >= 0) & (valid_timelines < 1500)]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(valid_timelines, bins=60, color=sns.color_palette("muted")[2], edgecolor="white")
    ax.axvline(valid_timelines.median(), color="red", linestyle="--", label=f"Median: {valid_timelines.median():.0f} days")
    ax.set_title("ADU Permit Approval Timeline (Application → Issue)", fontsize=14)
    ax.set_xlabel("Days")
    ax.set_ylabel("Number of permits")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Median: {valid_timelines.median():.0f} days | Mean: {valid_timelines.mean():.0f} days | 90th pct: {valid_timelines.quantile(0.9):.0f} days")

In [ ]:
# Value of work distribution
value_col = next((c for c in df_adu.columns if "value" in c.lower() or "estvalue" in c.lower()), None)

if value_col:
    df_adu["value_clean"] = pd.to_numeric(df_adu[value_col], errors="coerce")
    values = df_adu["value_clean"].dropna()
    values = values[(values > 0) & (values < values.quantile(0.99))]  # trim top 1% outliers

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(values / 1000, bins=60, color=sns.color_palette("muted")[3], edgecolor="white")
    ax.axvline(values.median() / 1000, color="red", linestyle="--", label=f"Median: ${values.median()/1000:.0f}k")
    ax.set_title("ADU Permit — Estimated Value of Work", fontsize=14)
    ax.set_xlabel("Value ($thousands)")
    ax.set_ylabel("Number of permits")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Median: ${values.median():,.0f} | Mean: ${values.mean():,.0f} | 90th pct: ${values.quantile(0.9):,.0f}")

## 6. Summary of findings and data quality notes

Document what was found and flag issues for the dbt layer. Update this section after running the notebook.

In [ ]:
# Data quality issues to carry into dbt tests
# Update this after running the full notebook — these become the basis for dbt tests

data_quality_notes = {
    "high_null_columns": "See section 3 — columns with >50% nulls should not be used in NOT NULL tests",
    "date_parsing": "Some date fields contain unparseable values — cast carefully in staging",
    "value_outliers": "Estimated value field has extreme high-end outliers — consider capping or flagging",
    "adu_filter": f"ADU search terms used: {ADU_TERMS} — review matches manually to confirm precision",
    "year_filter": "Pre-2000 records appear to have data quality issues — filtered from trend analysis",
}

print("Data quality notes for dbt layer:")
for k, v in data_quality_notes.items():
    print(f"\n  [{k}]\n  {v}")